# Adjustable Robust Optimization

Static robust optimization protects against uncertainty by requiring feasibility for *every* realization before the uncertain parameters are revealed. This is often too conservative: in many practical settings some decisions can be **deferred** until after part of the uncertainty is observed.

**Adjustable robust optimization (ARO)** {cite:p}`BenTalGoryashko2004` models this two-stage structure explicitly:

| Stage | Variables | When decided | Must satisfy |
|---|---|---|---|
| 1 (here-and-now) | $x$ | Before observing $\xi$ | Constraints for *all* $\xi \in \mathcal{U}$ |
| 2 (wait-and-see) | $y(\xi)$ | After observing $\xi$ | Constraints for the *actual* $\xi$ |

The two-stage minimax problem is:

$$
\min_{x} \max_{\xi \in \mathcal{U}} \min_{y} \; f(x, y) \quad
\text{s.t.}\; g(x, y, \xi) \le 0
$$

## Affine Decision Rules

Optimizing over arbitrary functions $y(\xi)$ is generally intractable. The
**affine decision rule** (ADR) approximation {cite:p}`BenTalGoryashko2004` restricts
recourse to affine functions of the uncertainty:

$$
y(\xi) = y_0 + \sum_{j=1}^{n_\xi} Y_j \cdot \xi_j, \qquad
\xi_j = p_j - \bar{p}_j
$$

where $y_0$ (intercept) and $Y_j$ (policy columns) become ordinary decision
variables. The ADR is **optimal** for problems where the feasible set is
convex and uncertainty enters linearly {cite:p}`BenTalGoryashko2004`. For
general nonlinear recourse it is an inner approximation.

The power of affine rules over static robust optimization was studied by
{cite:t}`Bertsimas2010`: they show that for LP problems with uncertain
right-hand sides, affine rules are optimal; for uncertain constraint matrices
they can be suboptimal.

## discopt Pipeline

```
Nominal model  →  AffineDecisionRule.apply()  →  RobustCounterpart.formulate()
(has y, ξ)        (y → y₀ + ΣYⱼξⱼ;              (ξ → worst-case constant;
                   y retired from variables)        model is deterministic)
                        ↓
                  Deterministic model: optimize over (x, y₀, Y_cols)
```

In [ ]:
import numpy as np
import discopt.modeling as dm
from discopt.ro import AffineDecisionRule, BoxUncertaintySet, RobustCounterpart

## Example 1 — Two-Stage Inventory Management

A retailer places a **first-stage** order $x$ before observing demand, then
can place an expensive **second-stage** emergency order $y$ once demand $d$
is known. Demand is uncertain: $d \in [60, 100]$ with nominal $\bar{d} = 80$.

$$
\min_{x, y(d)}\; c_1 x + c_2 y \quad
\text{s.t.}\; x + y \ge d,\; 0 \le x \le 200,\; 0 \le y \le 100
$$

With the affine rule $y(d) = y_0 + Y_0 (d - \bar{d})$, the recourse adapts
linearly to demand deviations.

In [ ]:
m = dm.Model("inventory")

x = m.continuous("order", lb=0, ub=200)      # here-and-now
y = m.continuous("emergency", lb=0, ub=100)  # wait-and-see recourse

d_bar = 80.0
d = m.parameter("d", value=d_bar)

c1, c2 = 2.0, 8.0  # unit ordering costs (emergency much more expensive)

m.minimize(c1 * x + c2 * y)
m.subject_to(x + y >= d, name="demand")
m.subject_to(x + y <= 200, name="capacity")

print("Before ADR:")
print(f"  Variables: {[v.name for v in m._variables]}")
print(f"  Objective: {m._objective}")
print(f"  Demand constraint: {m._constraints[0]}")

In [ ]:
# Stage 1: substitute y with affine rule  y(ξ) = y₀ + Y₀·(d - 80)
adr = AffineDecisionRule(y, uncertain_params=d, prefix="adr")
adr.apply()

print("After AffineDecisionRule.apply():")
print(f"  Variables: {[v.name for v in m._variables]}")
print(f"  Intercept y₀: {adr.intercept.name}")
print(f"  Policy column Y₀: {adr.policy_columns[0].name}")
print(f"  Affine expression: {adr.affine_expression}")
print(f"  Demand constraint: {m._constraints[0]}")

In [ ]:
# Stage 2: handle remaining ξ = d - 80 with box robust counterpart
# Uncertainty: d ∈ [60, 100]  →  delta = 20
rc = RobustCounterpart(m, BoxUncertaintySet(d, delta=20.0))
rc.formulate()

print("After RobustCounterpart.formulate() — fully deterministic:")
print(f"  Variables: {[v.name for v in m._variables]}")
print(f"  Demand constraint: {m._constraints[0]}")
print(f"  Objective: {m._objective}")
print()
print(adr.summary())

The reformulated model now optimizes over $(x, y_0, Y_0)$:
- $x$ is the baseline order quantity
- $y_0$ is the baseline emergency order (when $d = \bar{d}$)
- $Y_0$ is the *responsiveness* of the emergency order to demand deviations

Compare this to the static robust counterpart, which would just use $d = 100$ everywhere.

## Example 2 — Production Planning: Nominal vs Static vs Adjustable

We compare three approaches on a simple LP:

$$
\min_{x, y}\; 3x + 5y \quad
\text{s.t.}\; x + y \ge d,\; x \le 50,\; y \ge 0
$$

with uncertain demand $d \in [30, 70]$ (nominal $\bar{d} = 50$).

For this linear problem the affine rule is **optimal** among all measurable
decision rules {cite:p}`BenTalGoryashko2004` — it achieves the same objective
as the fully flexible adjustable solution.

In [ ]:
def _demand_constraint_constants(m):
    """Extract all Constant values embedded in the first constraint."""
    from discopt.modeling.core import (
        BinaryOp, Constant, FunctionCall, IndexExpression,
        MatMulExpression, SumExpression, UnaryOp
    )

    def _collect(expr):
        if isinstance(expr, Constant): return [float(np.sum(expr.value))]
        if isinstance(expr, (BinaryOp, MatMulExpression)):
            return _collect(expr.left) + _collect(expr.right)
        if isinstance(expr, UnaryOp): return _collect(expr.operand)
        if isinstance(expr, FunctionCall):
            r = []
            for a in expr.args: r += _collect(a)
            return r
        if isinstance(expr, IndexExpression): return _collect(expr.base)
        if isinstance(expr, SumExpression): return _collect(expr.operand)
        return []

    return _collect(m._constraints[0].body)


# ── Approach 1: Nominal (ignore uncertainty) ──────────────────────────────
m_nom = dm.Model("nominal")
x_n = m_nom.continuous("x", lb=0, ub=50)
y_n = m_nom.continuous("y", lb=0)
d_n = m_nom.parameter("d", value=50.0)
m_nom.minimize(3 * x_n + 5 * y_n)
m_nom.subject_to(x_n + y_n >= d_n)
# No uncertainty handling: d stays at 50
print("Nominal:  demand constraint embedded constant(s):",
      _demand_constraint_constants(m_nom))

# ── Approach 2: Static robust counterpart ────────────────────────────────
m_stat = dm.Model("static_robust")
x_s = m_stat.continuous("x", lb=0, ub=50)
y_s = m_stat.continuous("y", lb=0)
d_s = m_stat.parameter("d", value=50.0)
m_stat.minimize(3 * x_s + 5 * y_s)
m_stat.subject_to(x_s + y_s >= d_s)
RobustCounterpart(m_stat, BoxUncertaintySet(d_s, delta=20.0)).formulate()
print("Static:   demand constraint embedded constant(s):",
      _demand_constraint_constants(m_stat))

# ── Approach 3: Adjustable robust (affine rule) ──────────────────────────
m_adj = dm.Model("adjustable_robust")
x_a = m_adj.continuous("x", lb=0, ub=50)
y_a = m_adj.continuous("y", lb=0)
d_a = m_adj.parameter("d", value=50.0)
m_adj.minimize(3 * x_a + 5 * y_a)
m_adj.subject_to(x_a + y_a >= d_a)
AffineDecisionRule(y_a, uncertain_params=d_a).apply()
RobustCounterpart(m_adj, BoxUncertaintySet(d_a, delta=20.0)).formulate()
print("Adjustable: variables after reformulation:",
      [v.name for v in m_adj._variables])

- **Nominal**: uses $d = 50$. Cheap but infeasible for $d > 50$.
- **Static robust**: uses $d = 70$ (worst case). Always feasible but overly conservative — orders more than necessary when $d < 70$.
- **Adjustable**: the emergency order $y$ responds to actual demand. Less conservative than static robust while still guaranteeing feasibility.

## Example 3 — Multi-Parameter Recourse

A recourse variable can respond to *multiple* uncertain parameters simultaneously.
Each uncertain parameter $p_i$ contributes one policy column $Y_i$:

$$
y(\xi) = y_0 + Y_c \cdot (c - \bar{c}) + Y_d \cdot (d - \bar{d})
$$

Here $c$ is an uncertain cost parameter and $d$ is uncertain demand.

In [ ]:
m = dm.Model("multi_param_recourse")

x = m.continuous("x", lb=0, ub=100)       # here-and-now
y = m.continuous("y", lb=0)               # wait-and-see

c = m.parameter("c", value=3.0)           # uncertain unit cost
d = m.parameter("d", value=40.0)          # uncertain demand

m.minimize(c * x + 2 * y)
m.subject_to(x + y >= d, name="demand")

# y adapts to both cost and demand deviations
adr = AffineDecisionRule(y, uncertain_params=[c, d], prefix="pol")
adr.apply()

print(f"Policy columns created: {[col.name for col in adr.policy_columns]}")
print(f"  pol_Y0 responds to: c - {c.value}  (cost deviation)")
print(f"  pol_Y1 responds to: d - {d.value}  (demand deviation)")
print()

# Apply static RO to both uncertain parameters
rc = RobustCounterpart(
    m,
    [BoxUncertaintySet(c, delta=0.5), BoxUncertaintySet(d, delta=10.0)]
)
rc.formulate()

print("Final model variables:")
for v in m._variables:
    print(f"  {v.name}  shape={v.shape}")

## Example 4 — Vector Recourse Variable

When the recourse variable is a vector $y \in \mathbb{R}^m$, each policy
column $Y_j$ is also a vector of the same shape, giving the element-wise
rule:

$$
y_k(\xi) = (y_0)_k + \sum_j (Y_j)_k \cdot \xi_j, \quad k = 1,\ldots,m
$$

In [ ]:
n_products = 4

m = dm.Model("vector_recourse")
x = m.continuous("x", shape=(n_products,), lb=0)
y = m.continuous("y", shape=(n_products,), lb=0)  # recourse vector

d = m.parameter("d", value=np.ones(n_products) * 20.0)  # uncertain demands

costs = np.array([3.0, 5.0, 4.0, 6.0])
m.minimize(dm.sum(costs * x) + dm.sum(2 * costs * y))
m.subject_to(x + y >= d, name="demand")

# Each of the 4 demand components contributes one policy column
adr = AffineDecisionRule(y, uncertain_params=d, prefix="vec")
adr.apply()

print(f"Recourse y has shape (4,); uncertain d has 4 components")
print(f"Policy columns created: {len(adr.policy_columns)}")
for i, col in enumerate(adr.policy_columns):
    print(f"  {col.name}  shape={col.shape}")

print(f"\nTotal new variables from ADR: {1 + len(adr.policy_columns)} × shape(4,)")
print(f"  = {(1 + len(adr.policy_columns)) * n_products} scalar decision variables")

## API Summary

```python
from discopt.ro import AffineDecisionRule, BoxUncertaintySet, RobustCounterpart

# 1. Build nominal two-stage model
m = dm.Model()
x = m.continuous("x", ...)                  # here-and-now
y = m.continuous("y", ...)                  # wait-and-see (recourse)
p = m.parameter("p", value=nominal)         # uncertain parameter
# ... build objective and constraints ...

# 2. Apply affine decision rule  y → y₀ + Y₀·(p - p̄)
adr = AffineDecisionRule(y, uncertain_params=p)   # or list of params
adr.apply()                                        # modifies m in-place

# 3. Apply static robust counterpart for remaining uncertainty
rc = RobustCounterpart(m, BoxUncertaintySet(p, delta=0.1 * nominal))
rc.formulate()                                     # modifies m in-place

# 4. Solve — decision variables are now (x, y₀, Y₀)
result = m.solve()
```

**After `apply()`:**
- `adr.intercept` — the $y_0$ variable
- `adr.policy_columns` — list of $Y_j$ variables
- `adr.affine_expression` — the full expression $y_0 + \sum_j Y_j \xi_j$
- `adr.perturbations` — list of perturbation expressions $\xi_j = p_j - \bar{p}_j$
- `adr.n_policy_columns` — total number of scalar uncertain components

## Theoretical Guarantees and Limitations

**Optimality** {cite:t}`BenTalGoryashko2004`: For LP problems with
uncertainty appearing only in the right-hand side, affine decision rules are
*optimal* — no other measurable recourse policy achieves a lower cost.

**Sub-optimality** {cite:t}`Bertsimas2010`: For uncertain constraint matrices
(i.e., $A(\xi)x \le b(\xi)$), affine rules may be sub-optimal. In these
cases they serve as a tractable inner approximation.

**Current limitations** in discopt's implementation:
- Recourse variables must be scalar or 1-D.
- The policy matrix scales as $\dim(y) \times n_\xi$ new scalar variables,
  which can be large for high-dimensional problems.
- Non-affine recourse (polynomial, piecewise-linear) is not yet supported.

## References

```{bibliography}
:filter: docname in docnames
```